<a href="https://colab.research.google.com/github/Dani-Mol/InteligenciaArtificial/blob/main/Clase_09_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 09-01: Merge Intervals

## 1. Manejo de Python

Para resolver este problema de manera óptima, no podemos usar matrices clásicas (arrays 2D) debido a los límites del problema ($n, m \le 10^9$). Representar una matriz de $10^9 \times 10^9$ consumiría demasiada memoria y nos daría un error por límite de tiempo y espacio.

En su lugar, utilizaremos las siguientes herramientas fundamentales de Python:
*   **Diccionarios (`collections.defaultdict`)**: Nos permiten agrupar las vías del tren por fila rápidamente sin tener que inicializar todas las filas de la cuadrícula, ahorrando memoria masivamente.
*   **Listas e Intervalos**: Representaremos las vías como listas de pares `[inicio, fin]`.
*   **Ordenamiento con funciones Lambda (`sorted` / `.sort()`)**: Es fundamental para el algoritmo de *fusión de intervalos* (Merge Intervals), ordenando las vías primero por su columna de inicio.

In [ ]:
from collections import defaultdict

# ==============================================================================
# MANEJO DE DICCIONARIOS CON DEFAULTDICT
# ==============================================================================
# El uso de defaultdict evita errores de 'KeyError' al acceder a llaves que no existen.
# Inicializa automáticamente una lista vacía si la llave (fila) es nueva.
vias_por_fila = defaultdict(list)
vias_por_fila[1].append([1, 4])
vias_por_fila[1].append([6, 9])

print('Vías en la fila 1:', vias_por_fila[1])


# ==============================================================================
# ORDENAMIENTO DE INTERVALOS Y CON FUNCIONES LAMBDA
# ==============================================================================
# En algoritmos como "Merge Intervals", procesar los datos de izquierda a derecha
# requiere que los tramos o intervalos estén estrictamente ordenados por su punto
# de inicio (la columna c1).

intervalos_desordenados = [[5, 8], [1, 4], [3, 6]]

# ------------------------------------------------------------------------------
# ¿Qué es una función Lambda?
# ------------------------------------------------------------------------------
# Una función `lambda` es una función anónima (sin nombre) de una sola línea.
# Sintaxis general: lambda argumentos: expresion_que_retorna
#
# La siguiente lambda toma un elemento 'x' (que será una lista como [5, 8])
# y regresa únicamente su primer elemento 'x[0]' (es decir, el 5).

extraer_inicio = lambda x: x[0]

# ------------------------------------------------------------------------------
# ¿Cómo funciona el parámetro 'key' en sorted() / .sort()?
# ------------------------------------------------------------------------------
# La función built-in `sorted()` no compara las listas directamente cuando le
# pasamos el parámetro `key`. En su lugar, aplica la función a cada elemento y
# usa los resultados devueltos para decidir el orden.
#
# Evaluación paso a paso que realiza Python internamente:
# 1. Toma [5, 8] -> evalúa lambda([5, 8]) -> devuelve 5
# 2. Toma [1, 4] -> evalúa lambda([1, 4]) -> devuelve 1
# 3. Toma [3, 6] -> evalúa lambda([3, 6]) -> devuelve 3
# 4. Ordena los elementos basándose en las llaves [5, 1, 3] -> resultado: 1, 3, 5
# 5. Reorganiza las listas originales: [[1, 4], [3, 6], [5, 8]]

intervalos_ordenados = sorted(intervalos_desordenados, key=lambda x: x[0])
print('Intervalos ordenados por columna de inicio (c1):', intervalos_ordenados)


# ------------------------------------------------------------------------------
# VARIACIONES ÚTILES DE LAMBDA EN ORDENAMIENTO
# ------------------------------------------------------------------------------

# A) Ordenar por el límite final (columna c2):
# Si quisiéramos ordenar por el segundo elemento en lugar del primero:
por_fin = sorted(intervalos_desordenados, key=lambda x: x[1])
print('Ordenados por columna de fin (c2):', por_fin)

# B) Criterio de ordenamiento múltiple (Desempate):
# Si dos intervalos empiezan en la misma columna, podemos usar una tupla en el lambda
# para desempatar por el final (de menor a mayor):
intervalos_con_empate = [[3, 8], [1, 4], [3, 5]]
por_inicio_y_fin = sorted(intervalos_con_empate, key=lambda x: (x[0], x[1]))
print('Ordenados por inicio y desempate por fin:', por_inicio_y_fin)

# C) Ordenamiento en reversa (descendente):
por_inicio_desc = sorted(intervalos_desordenados, key=lambda x: x[0], reverse=True)
print('Ordenados por inicio en forma descendente:', por_inicio_desc)

Vías en la fila 1: [[1, 4], [6, 9]]
Intervalos ordenados por columna de inicio (c1): [[1, 4], [3, 6], [5, 8]]
Ordenados por columna de fin (c2): [[1, 4], [3, 6], [5, 8]]
Ordenados por inicio y desempate por fin: [[1, 4], [3, 5], [3, 8]]
Ordenados por inicio en forma descendente: [[5, 8], [3, 6], [1, 4]]


## 2. Entendiendo el Problema

El reto **Gridland Metro** nos plantea una cuadrícula de dimensiones $n \times m$. En esta cuadrícula hay vías de tren dadas por una fila $r$, y un rango de columnas $[c_1, c_2]$.

Las vías pueden solaparse. Nuestro objetivo es encontrar el **número total de celdas libres** donde se pueden colocar farolas (celdas sin vías).

**Ejemplo de solapamiento:**
Si en la fila 1 tenemos una vía que va de la columna 1 a la 4, y en los datos nos informan de otra vía en la misma fila 1 que va de la columna 3 a la 6, en realidad todo el tramo se une formando una gran vía ininterrumpida de la columna 1 a la 6. Esto significa que ocupan 6 celdas, no 8.

## 3. Lógica y Estructura del Algoritmo **Merge Intervals**

1.  **Agrupación**: Guardar todas las vías en un diccionario donde la clave es el número de fila $r$, y el valor es una lista de intervalos `[c1, c2]`.
2.  **Ordenar**: Iterar sobre las claves del diccionario. Para cada fila, ordenar sus intervalos por el punto de inicio `c1` de menor a mayor.
3.  **Fusión**: Recorrer los intervalos de la fila de izquierda a derecha. Si el inicio de un intervalo se solapa con el fin del intervalo anterior, los combinamos extendiendo el límite derecho del previo. Si no se solapan, lo agregamos como un nuevo bloque separado.
4.  **Cálculo**: Sumar la longitud de todos los bloques fusionados resultantes. Finalmente, se resta este total del área total de la malla($n \times m$).

In [ ]:
def fusionar_intervalos(intervalos):

    if not intervalos:
        return []

    # Ordenamos por el inicio del intervalo
    intervalos.sort(key=lambda x: x[0])

    fusionados = [intervalos[0]]
    for actual in intervalos[1:]:
        previo = fusionados[-1]
        # Si hay solapamiento (el inicio del actual es menor o igual al fin del previo)
        if actual[0] <= previo[1]:
            # Extendemos el fin del intervalo previo al máximo alcance
            previo[1] = max(previo[1], actual[1])
        else:
            # No hay solapamiento, agregamos como nuevo intervalo independiente
            fusionados.append(actual)

    return fusionados

# Prueba de la lógica de fusión:
ejemplo = [[1, 5], [2, 3], [6, 8], [7, 10]]
print("Intervalos Originales:", ejemplo)
print("Intervalos Fusionados:", fusionar_intervalos(ejemplo))

Intervalos Originales: [[1, 5], [2, 3], [6, 8], [7, 10]]
Intervalos Fusionados: [[1, 5], [6, 10]]


# Exploración de Algoritmos de Búsqueda: Dijkstra vs A*

Este cuaderno explora dos de los algoritmos fundamentales para la búsqueda de rutas óptimas en grafos. Presenta una estructura ideal para visualizar y comprender conceptos clave de Inteligencia Artificial, mostrando la transición lógica y matemática desde una búsqueda exhaustiva de costo uniforme hacia una búsqueda informada y heurística.

Utilizaremos `collections.defaultdict` de manera intensiva. Su aplicación no solo simplifica la creación de la lista de adyacencia del grafo, sino que también nos permite gestionar dinámicamente los registros de distancias inicializándolos en infinito de forma automática.

In [ ]:
def dijkstra(grafo, inicio, objetivo):
    # Uso avanzado de defaultdict para las distancias.
    # Si consultamos un nodo no visitado, automáticamente retorna infinito.
    distancias = defaultdict(lambda: float('inf'))
    distancias[inicio] = 0

    # Diccionario para rastrear el árbol de expansión y reconstruir el camino
    padres = {}

    # Cola de prioridad: almacena tuplas de (costo_acumulado, nodo_actual)
    pq = [(0, inicio)]

    # Métrica de eficiencia para comparar más adelante
    nodos_visitados = 0

    while pq:
        costo_actual, nodo_actual = heapq.heappop(pq)
        nodos_visitados += 1

        # Condición de parada: hemos alcanzado y consolidado el objetivo
        if nodo_actual == objetivo:
            break

        # Poda: Si extraemos un camino más largo o anticuado al ya registrado en el diccionario, lo ignoramos
        if costo_actual > distancias[nodo_actual]:
            continue

        # Expansión
        for vecino, peso in grafo[nodo_actual]:
            nuevo_costo = costo_actual + peso

            # Relajación de la arista
            if nuevo_costo < distancias[vecino]:
                distancias[vecino] = nuevo_costo
                padres[vecino] = nodo_actual
                heapq.heappush(pq, (nuevo_costo, vecino))

    # Reconstrucción de la ruta trazando los padres hacia atrás
    camino = []
    actual = objetivo
    while actual in padres:
        camino.append(actual)
        actual = padres[actual]
    camino.append(inicio)
    camino.reverse()

    return camino, distancias[objetivo], nodos_visitados

camino_d, costo_d, visitados_d = dijkstra(grafo, 'A', 'D')
print(f"Dijkstra -> Ruta: {camino_d} | Costo total: {costo_d} | Nodos expandidos: {visitados_d}")

Dijkstra -> Ruta: ['A', 'B', 'C', 'D'] | Costo total: 4 | Nodos expandidos: 5
